# Week 5 Day 2 - Pydantic AI exercise

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

import board

load_dotenv(override=True)

In [ ]:
MODEL = "openai-chat:gpt-5.4-mini"

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [ ]:
workspace = Path("workspace").resolve()   # the only folder the agent may touch

filesystem = MCPToolset(
    StdioTransport(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
        cwd=str(workspace),  # start the server in the workspace so relative file names resolve there
        log_file=Path(os.devnull),
    ), 
    init_timeout=60
)

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    MODEL,
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task],
    toolsets=[filesystem],
)

board.reset_board()
goal_id = board.add_goal("Write a short haiku about Madrid into madrid.txt.")
board.claim_todo(goal_id)

async with worker:
    result = await worker.run("Please work the pending goal on the board.")
print(result.output)
board.show_board()

## Inspect the full transcript

In [ ]:
for message in result.all_messages():
    for part in message.parts:
        print(type(part).__name__, '->', part)
    print('-' * 40)